# 03. Avaliar linkage

Pares do [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb) (`predict` grava
`p ≥ 0,5`). Corte operacional `T = THRESHOLD_AVALIACAO` (default 0,99).

1. Exemplos com `p ≥ T`
2. Exemplos na faixa `[T − 0,05, T)`
3. Melhor CPF por Censo acima de T; CPFs únicos vs compartilhado
4. Recall ouro: par 1:1 da coorte que cai numa associação única
5. Pares com `p < T` que já são 1:1 no grafo (1 Censo ↔ 1 CPF)

Sem cluster. Lista operacional (só as únicas) no [`04_atribuir.ipynb`](04_atribuir.ipynb).

A ouro é amostra 1:1 da `cohort_dedup` no recorte, não o universo.


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T = THRESHOLD_AVALIACAO
T_FAIXA = T - 0.05
TOP_N = 20

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('T:', T, '| faixa >=', T_FAIXA, 'e <', T)


Predictions no parquet (ids + score). Ouro 1:1 no recorte.


In [ ]:
cols_pred = list(
    con.execute(f"SELECT * FROM read_parquet('{SPLINK_PREDICTIONS}') LIMIT 0").df().columns
)
tem_weight = 'match_weight' in cols_pred
col_weight = ', match_weight' if tem_weight else ''
score_sql = 'p.match_probability'
if tem_weight:
    score_sql = 'p.match_probability, p.match_weight'
cols_exemplo = '''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.uf AS uf_censo,
    pb.uf AS uf_cpf,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf
'''

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
    {col_weight}
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

n_pares = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print('Pares no parquet:', f'{n_pares:,}')

counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Ouro 1:1 no subset:', f'{n_gt:,}')
print('Não 1:1 descartados (N:1 / 1:N):', f"{counts['n_nao_1a1_descartada']:,}")
if n_gt == 0:
    raise RuntimeError(
        'Nenhum par da coorte caiu no subset — confira o filtro geográfico do NB00.'
    )


## Pares com p ≥ T

Candidatos acima do corte operacional.


In [ ]:
n_acima = con.execute(f'''
SELECT COUNT(*) FROM splink_predictions WHERE match_probability >= {T}
''').fetchone()[0]
print(f'Pares com p >= {T}:', f'{n_acima:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    {score_sql},
    {cols_exemplo}
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T}
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


## Faixa [T − 0,05, T)

Pares perto do corte, ainda abaixo de T.


In [ ]:
n_faixa = con.execute(f'''
SELECT COUNT(*)
FROM splink_predictions
WHERE match_probability >= {T_FAIXA}
  AND match_probability < {T}
''').fetchone()[0]
print(f'Pares com {T_FAIXA} <= p < {T}:', f'{n_faixa:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    {score_sql},
    {cols_exemplo}
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T_FAIXA}
  AND p.match_probability < {T}
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


## Melhor CPF por Censo (p ≥ T)

Cada Censo fica com o CPF de maior `match_probability` (empate: `unique_id_cpf`).
Não há greedy nem cluster. CPF com dois ou mais Censos no topo entra na conta
de múltiplos; associação única = esse CPF aparece uma vez.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
WHERE match_probability >= {T}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute('''
CREATE OR REPLACE TABLE cpf_n_censo AS
SELECT unique_id_cpf, COUNT(*) AS n_censo
FROM melhor_por_censo
GROUP BY 1
''')

con.execute('''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT m.*
FROM melhor_por_censo m
JOIN cpf_n_censo c ON c.unique_id_cpf = m.unique_id_cpf
WHERE c.n_censo = 1
''')

display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM splink_predictions WHERE match_probability >= {T})
        AS n_pares_acima_t,
    (SELECT COUNT(*) FROM melhor_por_censo) AS n_censos_com_par,
    (SELECT COUNT(*) FROM associacoes_unicas) AS n_associacoes_unicas,
    (SELECT COUNT(*) FROM cpf_n_censo WHERE n_censo = 1) AS n_cpf_unicos,
    (SELECT COUNT(*) FROM cpf_n_censo WHERE n_censo >= 2) AS n_cpf_multiplos
''').df())


In [ ]:
display(con.execute(f'''
SELECT
    m.unique_id_cpf,
    c.n_censo,
    m.unique_id_censo,
    m.match_probability,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf
FROM melhor_por_censo m
JOIN cpf_n_censo c ON c.unique_id_cpf = m.unique_id_cpf
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE c.n_censo >= 2
ORDER BY c.n_censo DESC, m.unique_id_cpf, m.match_probability DESC
LIMIT {TOP_N}
''').df())


## Recall ouro

Universo: pares 1:1 da coorte no recorte. Achado: `(Censo, CPF)` ouro está nas
associações únicas (melhor nota e o CPF não é de outro Censo).


In [ ]:
display(con.execute(f'''
SELECT
    (SELECT COUNT(*) FROM gt_no_subset) AS n_ouro,
    (
        SELECT COUNT(*)
        FROM gt_no_subset gt
        JOIN associacoes_unicas u
          ON u.unique_id_censo = gt.unique_id_censo
         AND u.unique_id_cpf = gt.unique_id_cpf
    ) AS n_encontrados,
    ROUND(
        100.0 * (
            SELECT COUNT(*)
            FROM gt_no_subset gt
            JOIN associacoes_unicas u
              ON u.unique_id_censo = gt.unique_id_censo
             AND u.unique_id_cpf = gt.unique_id_cpf
        ) / NULLIF((SELECT COUNT(*) FROM gt_no_subset), 0),
        2
    ) AS recall_pct,
    (
        SELECT COUNT(DISTINCT gt.unique_id_cpf)
        FROM gt_no_subset gt
        JOIN associacoes_unicas u ON u.unique_id_cpf = gt.unique_id_cpf
    ) AS n_cpf_ouro_nas_unicas
''').df())


## 1:1 abaixo de T

No recorte `p < T` (o parquet já é `p ≥ 0,5`): Censo com exatamente um CPF e
esse CPF com exatamente um Censo. Sem escolher o melhor score.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE um_para_um_abaixo AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM (
    SELECT
        unique_id_censo,
        unique_id_cpf,
        match_probability,
        COUNT(*) OVER (PARTITION BY unique_id_censo) AS n_cpf,
        COUNT(*) OVER (PARTITION BY unique_id_cpf) AS n_censo
    FROM splink_predictions
    WHERE match_probability < {T}
)
WHERE n_cpf = 1 AND n_censo = 1
''')

n_1a1 = con.execute('SELECT COUNT(*) FROM um_para_um_abaixo').fetchone()[0]
print(f'Pares 1:1 com p < {T}:', f'{n_1a1:,}')


In [ ]:
display(con.execute(f'''
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    {cols_exemplo}
FROM um_para_um_abaixo p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
ORDER BY p.match_probability DESC, p.unique_id_censo
LIMIT {TOP_N}
''').df())


In [ ]:
con.close()
